### SWDA rebalancing

The purpose of this notebook is to evaluate a rebalancing of a stock portfolio having SWDA etf. The purpose is to find the weight of the XMMI etf that makes it as close to VWCE as possible

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from pathlib import Path
import requests
from datetime import datetime
import openpyxl

from src.vwce.swda_xls_to_csv import convert
from src.optimization.optimization_utils import compute_optimal_weights

In [2]:
# Download SWDA allocation file from iShares website

url = (
    "https://www.ishares.com/uk/individual/en/products/251882/ishares-msci-world-ucits-etf-acc-fund/1535604580409.ajax?fileType=xls&fileName=iShares-Core-MSCI-World-UCITS-ETF_fund&dataType=fund"
)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://www.ishares.com/uk/individual/en/products/251882/",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

response = requests.get(url, headers=headers, stream=True)
response.raise_for_status()

# Nome file con data odierna
today = datetime.today().strftime("%Y-%m-%d")
filename = "../portfolio_allocations/SWDA_allocation.xls"

with open(filename, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

print(f"File salvato: {filename}")


File salvato: ../portfolio_allocations/SWDA_allocation.xls


In [3]:
# VWCE is different: it doesn't have a direct excel download but instead i have to convert a json file to xlsx

url = "https://www.nl.vanguard/gpx/graphql"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36",
    "Content-Type": "application/json",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "it-IT,it;q=0.9,en-US;q=0.8,en;q=0.7",
    "Origin": "https://www.nl.vanguard",
    "Referer": "https://www.nl.vanguard/professional/product/etf/equity/9679/ftse-all-world-ucits-etf-usd-accumulating",
    "apollographql-client-name": "gpx",
    "x-consumer-id": "nl0",
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "same-origin",
}

payload = {
    "operationName": "MarketAllocationGqlQuery",
    "variables": {"portIds": ["9679"]},
    "query": """query MarketAllocationGqlQuery($portIds: [String!]!) {
  funds(portIds: $portIds) {
    profile {
      fundFullName
      primaryMarketEquityClassification
      polarisPdtTypeIndicator
      marketOfDomicile
      __typename
    }
    marketAllocation {
      portId
      date
      countryCode
      countryName
      fundMktPercent
      holdingStatCode
      benchmarkMktPercent
      regionCode
      regionName
      __typename
    }
    __typename
  }
}
"""
}

response = requests.post(url, headers=headers, json=payload)
response.raise_for_status()

data = response.json()
allocations = data["data"]["funds"][0]["marketAllocation"]

# Tieni solo i dati FTSE (esclude duplicati MSCI)
allocations = [a for a in allocations if a["holdingStatCode"].startswith("FT")]


# Salva in Excel
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Market Allocation"

ws.append(["countryCode", "countryName", "regionCode", "regionName",
           "fundMktPercent", "benchmarkMktPercent", "date"])

for item in allocations:
    ws.append([
        item["countryCode"],
        item["countryName"],
        item["regionCode"],
        item["regionName"],
        item["fundMktPercent"],
        item["benchmarkMktPercent"],
        item["date"],
    ])

filename = "../portfolio_allocations/VWCE_allocation.xls"
wb.save(filename)
print(f"File salvato: {filename} ({len(allocations)} paesi)")

File salvato: ../portfolio_allocations/VWCE_allocation.xls (50 paesi)


In [4]:
# Download XMME allocation file from iShares website

url = (
    "https://etf.dws.com/etfdata/export/GBR/ENG/excel/product/constituent/IE00BTJRMP35/"
)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36",
    "Accept": "*/*",
    "Accept-Language": "it-IT,it;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://etf.dws.com/",
}

response = requests.get(url, headers=headers, stream=True)
response.raise_for_status()

# Nome file con data odierna
today = datetime.today().strftime("%Y-%m-%d")
filename = "../portfolio_allocations/XMME_allocation.xls"

with open(filename, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

print(f"File salvato: {filename}")





File salvato: ../portfolio_allocations/XMME_allocation.xls


In [5]:
# convert SWDA allocation file from xls to csv
input_path = Path('../portfolio_allocations/SWDA_allocation.xls')
output_path = Path('../portfolio_allocations/SWDA_allocation.csv')

convert(input_path, output_path)



1355

In [6]:
#get portfolio geographical info for SWDA
SWDA_alloc = pd.read_csv('../portfolio_allocations/SWDA_allocation.csv')

SWDA_alloc = SWDA_alloc.rename(columns={'Weight (%)': 'Weight'})
SWDA_alloc_geo = SWDA_alloc.groupby('Location')['Weight'].sum()
SWDA_alloc_geo = SWDA_alloc_geo.rename(index = {'--': 'Other'})
SWDA_alloc_geo

swda_countries = SWDA_alloc_geo.sort_index().index.tolist()

In [7]:
# get portfolio geographical info for XMME
df_temp = pd.read_excel('../portfolio_allocations/XMME_allocation.xlsx', header=None)

header_row_index = df_temp.index[df_temp.eq("Name").any(axis=1)].tolist()[0]

XMME_alloc = pd.read_excel('../portfolio_allocations/XMME_allocation.xlsx', skiprows=header_row_index, index_col = 0)

XMME_alloc['Weighting'] = XMME_alloc['Weighting'].astype(float) * 100
XMME_alloc = XMME_alloc.rename(columns={'Weighting': 'Weight (%)'})
XMME_alloc_geo = XMME_alloc.groupby('Country')['Weight (%)'].sum()

XMME_alloc_geo = XMME_alloc_geo.rename(index = {'-': 'Other'})
xmme_countries = XMME_alloc_geo.sort_index().index.tolist()

In [8]:
XMME_alloc_geo

Country
Other                    0.260057
Australia                0.039504
Brazil                   4.691774
Canada                   0.048711
Cayman Islands           0.070450
Chile                    0.533090
China                   18.951075
Colombia                 0.155011
Czech Republic           0.126489
Egypt                    0.058190
Greece                   0.475083
Hong Kong                3.530497
Hungary                  0.371443
India                   12.314220
Indonesia                0.834599
Ireland                  0.631357
Kazakhstan               0.018352
Korea, Republic of      17.615902
Kuwait                   0.596876
Luxembourg               0.064072
Malaysia                 1.136901
Mexico                   1.942198
Netherlands              0.105382
Peru                     0.242429
Philippines              0.324182
Poland                   1.110515
Qatar                    0.566301
Russian Federation       0.000002
Saudi Arabia             2.684474
Singap

In [9]:
#get portfolio geographical info for VWCE -> it already is allocated by Country


VWCE_alloc_geo = pd.read_excel('../portfolio_allocations/VWCE_allocation.xlsx', header = 0)
VWCE_alloc_geo = VWCE_alloc_geo.rename(columns={'fundMktPercent': 'Weight'})
VWCE_alloc_geo = VWCE_alloc_geo.reset_index().set_index('countryName')['Weight']
vwce_countries = VWCE_alloc_geo.sort_index().index.to_list()
VWCE_alloc_geo.index.name = 'Country'


In [10]:
VWCE_alloc_geo

Country
United Arab Emirates     0.17764
Austria                  0.07886
Australia                1.68080
Belgium                  0.25757
Brazil                   0.54260
Canada                   3.15994
Switzerland              2.10850
Chile                    0.07771
China                    3.18372
Colombia                 0.02320
Czech Republic           0.01441
Germany                  1.96560
Denmark                  0.34835
Egypt                    0.00494
Spain                    0.85649
Finland                  0.26774
France                   2.18411
United Kingdom           3.53055
Greece                   0.07458
Hong Kong                0.52713
Hungary                  0.04028
Indonesia                0.10252
Ireland                  0.08025
Israel                   0.32540
India                    1.64123
Iceland                  0.01050
Italy                    0.78397
Japan                    5.87150
Korea                    1.80392
Kuwait                   0.07662
Me

In [11]:
VWCE_alloc_geo

Country
United Arab Emirates     0.17764
Austria                  0.07886
Australia                1.68080
Belgium                  0.25757
Brazil                   0.54260
Canada                   3.15994
Switzerland              2.10850
Chile                    0.07771
China                    3.18372
Colombia                 0.02320
Czech Republic           0.01441
Germany                  1.96560
Denmark                  0.34835
Egypt                    0.00494
Spain                    0.85649
Finland                  0.26774
France                   2.18411
United Kingdom           3.53055
Greece                   0.07458
Hong Kong                0.52713
Hungary                  0.04028
Indonesia                0.10252
Ireland                  0.08025
Israel                   0.32540
India                    1.64123
Iceland                  0.01050
Italy                    0.78397
Japan                    5.87150
Korea                    1.80392
Kuwait                   0.07662
Me

In [12]:
vwce_countries = set(vwce_countries)
swda_countries = set(swda_countries)
xmme_countries = set(xmme_countries)

swda_xmme_countries = swda_countries.union(xmme_countries)

print('Countries in SWDA and XMME:', swda_xmme_countries)
print('Countries in VWCE and not in SWDA/XMME:', vwce_countries - swda_xmme_countries)     
print('Countries in SWDA/XMME and not in VWCE:', swda_xmme_countries - vwce_countries)
print('Countries in all three:', swda_xmme_countries.intersection(vwce_countries))


Countries in SWDA and XMME: {'Netherlands', 'Italy', 'Cayman Islands', 'Czech Republic', 'Taiwan', 'China', 'India', 'Denmark', 'Kuwait', 'Saudi Arabia', 'Brazil', 'Sweden', 'Other', 'Austria', 'Indonesia', 'Egypt', 'European Union', 'Greece', 'Chile', 'Thailand', 'Kazakhstan', 'Russian Federation', 'Finland', 'France', 'Norway', 'Canada', 'Australia', 'Colombia', 'Hungary', 'Switzerland', 'Ireland', 'Israel', 'Philippines', 'Luxembourg', 'South Africa', 'Mexico', 'Peru', 'United States', 'United Arab Emirates', 'Turkey', 'Poland', 'Germany', 'Singapore', 'Hong Kong', 'Spain', 'Korea, Republic of', 'Qatar', 'Malaysia', 'Belgium', 'Japan', 'Portugal', 'United Kingdom', 'New Zealand'}
Countries in VWCE and not in SWDA/XMME: {'Korea', 'Iceland', 'Russia', 'Romania'}
Countries in SWDA/XMME and not in VWCE: {'European Union', 'Cayman Islands', 'Korea, Republic of', 'Peru', 'Kazakhstan', 'Russian Federation', 'Luxembourg'}
Countries in all three: {'Netherlands', 'Italy', 'Czech Republic', 'T

In [13]:
VWCE_alloc_geo = VWCE_alloc_geo.rename(index={'United States of America': 'United States'})
XMME_alloc_geo = XMME_alloc_geo.rename(index={'Korea, Republic of': 'Korea'})

if "South korea" in XMME_alloc_geo.index:
    XMME_alloc_geo = XMME_alloc_geo.rename(index={'South korea': 'Korea'})
if "South Korea" in SWDA_alloc_geo.index:
    SWDA_alloc_geo = SWDA_alloc_geo.rename(index={'South Korea': 'Korea'})
if "South korea" in VWCE_alloc_geo.index:
    VWCE_alloc_geo = VWCE_alloc_geo.rename(index={'South korea': 'Korea'})

if "Russian Federation" in XMME_alloc_geo.index:
    XMME_alloc_geo = XMME_alloc_geo.rename(index={"Russian Federation": 'Russia'})
if "Russian Federation" in SWDA_alloc_geo.index:
    SWDA_alloc_geo = SWDA_alloc_geo.rename(index={"Russian Federation": 'Russia'})
if "Russian Federation" in VWCE_alloc_geo.index:
    VWCE_alloc_geo = VWCE_alloc_geo.rename(index={"Russian Federation": 'Russia'})

In [14]:
df_etf = pd.DataFrame({
    'SWDA': SWDA_alloc_geo,
    'XMME': XMME_alloc_geo,
    'VWCE': VWCE_alloc_geo
}).fillna(0).sort_values( by = 'VWCE', ascending = False)
df_etf.round(3).head(10)

,SWDA,XMME,VWCE
United States,71.320,0.529,61.337
Japan,5.737,0.000,5.872
United Kingdom,3.759,0.456,3.531
China,0.000,18.951,3.184
Canada,3.495,0.049,3.160
Taiwan,0.000,23.374,2.607
France,2.539,0.000,2.184
Switzerland,2.319,0.025,2.108
Germany,2.280,0.000,1.966
Korea,0.000,17.616,1.804


Let $\alpha, \beta$ be the vectors of allocation of the SWDA and of the XMME etfs, and let $\gamma$ be the vector of allocation of the VWCE etf, we want to minimize the following function

$$
f\colon \mathbb R^2 \to \mathbb R\\
(x,y)\mapsto ||(\alpha, \beta)\cdot (x,y)^\top - \gamma||^2
$$

which is convex. We use a standard optimizator for this kind of function

In [15]:
optimal_weights = compute_optimal_weights(df_etf[['SWDA', 'XMME']], df_etf['VWCE'])
print(optimal_weights)

Iteration 1: 36556763664.27566
Iteration 2: 2.3393832536867403
Optimization terminated successfully    (Exit mode 0)
            Current function value: 2.3393832536867403
            Iterations: 2
            Function evaluations: 10
            Gradient evaluations: 2
[0.86338699 0.13661301]


In [16]:
balanced_allocation = df_etf[['SWDA', 'XMME']].dot(optimal_weights)

comparison_df = pd.DataFrame({
    'SWDA+XMME': balanced_allocation,
    'VWCE': df_etf['VWCE']
})

comparison_df.round(3).sort_values(by='VWCE', ascending=False).head(10)

,SWDA+XMME,VWCE
United States,61.649,61.337
Japan,4.953,5.872
United Kingdom,3.308,3.531
China,2.589,3.184
Canada,3.024,3.160
Taiwan,3.193,2.607
France,2.192,2.184
Switzerland,2.006,2.108
Germany,1.969,1.966
Korea,2.407,1.804


In [17]:
print(optimal_weights)

[0.86338699 0.13661301]


In [18]:
# authentication to google APIs
from src.auth.google_credentials import load_google_credentials_dict

scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
creds = ServiceAccountCredentials.from_json_keyfile_dict(load_google_credentials_dict(), scope)
client = gspread.authorize(creds)

spreadsheet = client.open("SWDA-XMME_periodic_rebalancing")
sheet = spreadsheet.sheet1

In [19]:
# Clear old data before writing
sheet.batch_clear(["D2:F1000"])


# Update optimal weights in the sheet
sheet.update_acell('A2', optimal_weights[0])
sheet.update_acell('B2', optimal_weights[1])

# Update geographical allocations in the sheet
comparison_df = comparison_df.sort_values(by = 'VWCE', ascending=False)
N = comparison_df.shape[0]

country_list = comparison_df.index.tolist()
country_list = [[x] for x in country_list]

balanced_portfolio = comparison_df['SWDA+XMME'].values.tolist()
balanced_portfolio = [[x] for x in balanced_portfolio]

target_portfolio = comparison_df['VWCE'].values.tolist()
target_portfolio = [[x] for x in target_portfolio]


cell_interval = 'D2:D' + str(N+3)
sheet.update(country_list, cell_interval)

cell_interval = 'E2:E' + str(N+3)
sheet.update(balanced_portfolio, cell_interval)

cell_interval = 'F2:F' + str(N+3)
sheet.update(target_portfolio, cell_interval)


{'spreadsheetId': '1bBrqZe3lpPk7-80an_ueNTqEgqLV8q1CvGjPSfj28J8',
 'updatedRange': 'Foglio1!F2:F56',
 'updatedRows': 55,
 'updatedColumns': 1,
 'updatedCells': 55}